# Proyecto Sello: Monitor de Tanques de Agua — Fundamentos PySpark **Equipo Horeb**

## 1. Crear la SparkSessionPunto de entrada a Spark. Corre en modo local usando todos los núcleos disponibles.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("monitor-agua-horeb")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark

In [17]:
from pyspark.sql.types import (StructType, StructField, StringType,
    DoubleType, IntegerType)

# Definimos el esquema columna por columna
esquema = StructType([
    StructField("device_id",         StringType(),  True),
    StructField("timestamp",         StringType(),  True),
    StructField("distancia_cm",      DoubleType(),  True),
    StructField("nivel_porcentaje",  DoubleType(),  True),
    StructField("litros_actuales",   IntegerType(), True),
    StructField("capacidad_litros",  IntegerType(), True),
    StructField("bateria",           IntegerType(), True),
    StructField("zona",              StringType(),  True),
    StructField("ip",                StringType(),  True),
])

df = spark.read.csv(
    "../data/lecturas_tanques.csv",
    header=True,
    schema=esquema,
)

print("Filas:", df.count(), "| Columnas:", len(df.columns))

Filas: 50 | Columnas: 9


In [19]:
import os
print("Estoy en:", os.getcwd())
print("Archivos aquí:", os.listdir("."))
# Si existe la carpeta data, ver qué hay dentro:
if os.path.exists("../data"):
    print("Dentro de data/:", os.listdir("../data"))
else:
    print("⚠️ No existe la carpeta 'data' en esta ubicación")

Estoy en: /home/jovyan/Notebooks
Archivos aquí: ['.ipynb_checkpoints', 'monitor_agua_pyspark.ipynb']
Dentro de data/: ['lecturas_tanques.csv']


In [20]:
df.printSchema()

root
 |-- device_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- nivel_porcentaje: double (nullable = true)
 |-- litros_actuales: integer (nullable = true)
 |-- capacidad_litros: integer (nullable = true)
 |-- bateria: integer (nullable = true)
 |-- zona: string (nullable = true)
 |-- ip: string (nullable = true)



In [21]:
df.show(10, truncate=False)

+---------+-------------------+------------+----------------+---------------+----------------+-------+------+-------------+
|device_id|timestamp          |distancia_cm|nivel_porcentaje|litros_actuales|capacidad_litros|bateria|zona  |ip           |
+---------+-------------------+------------+----------------+---------------+----------------+-------+------+-------------+
|tanque_01|2026-08-20 06:00:00|14.3        |85.7            |857            |1000            |89     |Zona A|192.168.1.101|
|tanque_01|2026-08-20 08:07:00|18.2        |81.8            |818            |1000            |85     |Zona A|192.168.1.101|
|tanque_01|2026-08-20 10:21:00|25.6        |74.4            |744            |1000            |99     |Zona A|192.168.1.101|
|tanque_01|2026-08-20 12:02:00|32.2        |67.8            |678            |1000            |83     |Zona A|192.168.1.101|
|tanque_01|2026-08-20 14:00:00|35.7        |64.3            |643            |1000            |89     |Zona A|192.168.1.101|
|tanque_

In [22]:
df.select("nivel_porcentaje", "litros_actuales", "bateria").describe().show()

+-------+-----------------+------------------+------------------+
|summary| nivel_porcentaje|   litros_actuales|           bateria|
+-------+-----------------+------------------+------------------+
|  count|               49|                49|                49|
|   mean|59.23469387755101| 755.0816326530612| 91.24489795918367|
| stddev|17.23271538649196|369.33977337578864|5.1215663792311386|
|    min|             15.2|               152|                83|
|    max|             85.9|              1719|                99|
+-------+-----------------+------------------+------------------+



In [23]:
from pyspark.sql.functions import col

df_critico = (
    df
    .select("device_id", "timestamp", "nivel_porcentaje", "zona")
    .filter(col("nivel_porcentaje") < 30)
)

df_critico   # solo muestra el plan, no los datos

DataFrame[device_id: string, timestamp: string, nivel_porcentaje: double, zona: string]

In [24]:
df_critico.show(truncate=False)
print("Lecturas en nivel crítico:", df_critico.count())

+---------+-------------------+----------------+------+
|device_id|timestamp          |nivel_porcentaje|zona  |
+---------+-------------------+----------------+------+
|tanque_03|2026-08-21 00:06:00|15.2            |Zona C|
+---------+-------------------+----------------+------+

Lecturas en nivel crítico: 1


In [25]:
df_critico.explain(True)

== Parsed Logical Plan ==
'Filter ('nivel_porcentaje < 30)
+- Project [device_id#0, timestamp#1, nivel_porcentaje#3, zona#7]
   +- Relation [device_id#0,timestamp#1,distancia_cm#2,nivel_porcentaje#3,litros_actuales#4,capacidad_litros#5,bateria#6,zona#7,ip#8] csv

== Analyzed Logical Plan ==
device_id: string, timestamp: string, nivel_porcentaje: double, zona: string
Filter (nivel_porcentaje#3 < cast(30 as double))
+- Project [device_id#0, timestamp#1, nivel_porcentaje#3, zona#7]
   +- Relation [device_id#0,timestamp#1,distancia_cm#2,nivel_porcentaje#3,litros_actuales#4,capacidad_litros#5,bateria#6,zona#7,ip#8] csv

== Optimized Logical Plan ==
Project [device_id#0, timestamp#1, nivel_porcentaje#3, zona#7]
+- Filter (isnotnull(nivel_porcentaje#3) AND (nivel_porcentaje#3 < 30.0))
   +- Relation [device_id#0,timestamp#1,distancia_cm#2,nivel_porcentaje#3,litros_actuales#4,capacidad_litros#5,bateria#6,zona#7,ip#8] csv

== Physical Plan ==
*(1) Filter (isnotnull(nivel_porcentaje#3) AND (nive

In [26]:
from pyspark.sql.functions import when, lit, to_timestamp, hour

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df = df.withColumn(
    "estado",
    when(col("nivel_porcentaje") < 15, "CRITICO")
    .when(col("nivel_porcentaje") < 30, "BAJO")
    .when(col("nivel_porcentaje") < 60, "MEDIO")
    .otherwise("LLENO")
)

df = df.withColumn("hora", hour(col("timestamp")))

df = df.withColumn(
    "es_madrugada",
    when((col("hora") >= 0) & (col("hora") <= 5), lit(True)).otherwise(lit(False))
)

df = df.withColumn("proyecto", lit("Monitor Agua Horeb"))

df.select("device_id", "timestamp", "nivel_porcentaje", "estado",
          "hora", "es_madrugada").show(10, truncate=False)

+---------+-------------------+----------------+------+----+------------+
|device_id|timestamp          |nivel_porcentaje|estado|hora|es_madrugada|
+---------+-------------------+----------------+------+----+------------+
|tanque_01|2026-08-20 06:00:00|85.7            |LLENO |6   |false       |
|tanque_01|2026-08-20 08:07:00|81.8            |LLENO |8   |false       |
|tanque_01|2026-08-20 10:21:00|74.4            |LLENO |10  |false       |
|tanque_01|2026-08-20 12:02:00|67.8            |LLENO |12  |false       |
|tanque_01|2026-08-20 14:00:00|64.3            |LLENO |14  |false       |
|tanque_01|2026-08-20 16:16:00|57.7            |MEDIO |16  |false       |
|tanque_01|2026-08-20 18:06:00|50.4            |MEDIO |18  |false       |
|tanque_01|2026-08-20 20:13:00|46.0            |MEDIO |20  |false       |
|tanque_01|2026-08-20 22:25:00|37.8            |MEDIO |22  |false       |
|tanque_01|2026-08-21 00:22:00|37.4            |MEDIO |0   |true        |
+---------+-------------------+-------

In [27]:
from pyspark.sql.functions import count as spark_count, when as spark_when

df.select([
    spark_count(spark_when(col(c).isNull(), c)).alias(c)
    for c in ["nivel_porcentaje", "litros_actuales", "bateria"]
]).show()

+----------------+---------------+-------+
|nivel_porcentaje|litros_actuales|bateria|
+----------------+---------------+-------+
|               1|              1|      1|
+----------------+---------------+-------+



In [28]:
print("Filas con nivel_porcentaje nulo:")
df.filter(col("nivel_porcentaje").isNull()).select(
    "device_id", "timestamp", "nivel_porcentaje", "bateria").show(truncate=False)

df_limpio = df.filter(col("nivel_porcentaje").isNotNull())
print("Filas válidas tras limpiar:", df_limpio.count())

Filas con nivel_porcentaje nulo:
+---------+-------------------+----------------+-------+
|device_id|timestamp          |nivel_porcentaje|bateria|
+---------+-------------------+----------------+-------+
|tanque_04|2026-08-20 16:27:00|NULL            |87     |
+---------+-------------------+----------------+-------+

Filas válidas tras limpiar: 49


In [29]:
from pyspark.sql.functions import avg, min as spark_min, max as spark_max, count

resumen_tanque = df_limpio.groupBy("device_id").agg(
    avg("nivel_porcentaje").alias("nivel_promedio"),
    spark_min("nivel_porcentaje").alias("nivel_minimo"),
    spark_max("nivel_porcentaje").alias("nivel_maximo"),
    count("*").alias("num_lecturas")
)
resumen_tanque.show(truncate=False)

+---------+------------------+------------+------------+------------+
|device_id|nivel_promedio    |nivel_minimo|nivel_maximo|num_lecturas|
+---------+------------------+------------+------------+------------+
|tanque_05|63.03000000000001 |43.0        |85.9        |10          |
|tanque_01|60.329999999999984|37.4        |85.7        |10          |
|tanque_04|58.611111111111114|37.3        |84.7        |9           |
|tanque_03|54.38000000000001 |15.2        |85.0        |10          |
|tanque_02|59.760000000000005|38.4        |81.7        |10          |
+---------+------------------+------------+------------+------------+



In [30]:
resumen_zona = df_limpio.groupBy("zona").agg(
    avg("nivel_porcentaje").alias("nivel_promedio_zona"),
    avg("bateria").alias("bateria_promedio"),
    count("*").alias("lecturas")
)
resumen_zona.show(truncate=False)

+------+-------------------+-----------------+--------+
|zona  |nivel_promedio_zona|bateria_promedio |lecturas|
+------+-------------------+-----------------+--------+
|Zona A|59.515789473684194 |91.05263157894737|19      |
|Zona B|61.39500000000002  |91.3157894736842 |20      |
|Zona C|54.38000000000001  |91.9             |10      |
+------+-------------------+-----------------+--------+



In [31]:
df_limpio.groupBy("estado").agg(
    count("*").alias("cantidad")
).orderBy("cantidad", ascending=False).show()

+------+--------+
|estado|cantidad|
+------+--------+
| LLENO|      25|
| MEDIO|      23|
|  BAJO|       1|
+------+--------+



In [32]:
print("Lecturas en horario de madrugada (posibles fugas):")
df_limpio.filter(col("es_madrugada") == True).select(
    "device_id", "timestamp", "nivel_porcentaje", "zona", "estado"
).orderBy("nivel_porcentaje").show(truncate=False)

Lecturas en horario de madrugada (posibles fugas):
+---------+-------------------+----------------+------+------+
|device_id|timestamp          |nivel_porcentaje|zona  |estado|
+---------+-------------------+----------------+------+------+
|tanque_03|2026-08-21 00:06:00|15.2            |Zona C|BAJO  |
|tanque_04|2026-08-21 00:27:00|37.3            |Zona A|MEDIO |
|tanque_01|2026-08-21 00:22:00|37.4            |Zona A|MEDIO |
|tanque_02|2026-08-21 00:12:00|38.4            |Zona B|MEDIO |
|tanque_05|2026-08-21 00:15:00|43.0            |Zona B|MEDIO |
+---------+-------------------+----------------+------+------+



In [33]:
from operator import add

rdd = df_limpio.select("zona").rdd.map(lambda fila: fila.zona)
conteo_zonas = rdd.map(lambda z: (z, 1)).reduceByKey(add)

print("Conteo de lecturas por zona (con RDD):")
for zona, cantidad in conteo_zonas.collect():
    print(f"  {zona}: {cantidad} lecturas")

Conteo de lecturas por zona (con RDD):
  Zona A: 19 lecturas
  Zona B: 20 lecturas
  Zona C: 10 lecturas


In [34]:
niveles_bajos = (
    df_limpio.select("device_id", "nivel_porcentaje").rdd
    .filter(lambda f: f.nivel_porcentaje is not None and f.nivel_porcentaje < 40)
    .map(lambda f: (f.device_id, round(f.nivel_porcentaje, 1)))
)

print("Lecturas por debajo del 40% (device_id, nivel):")
for item in niveles_bajos.collect():
    print("  ", item)

Lecturas por debajo del 40% (device_id, nivel):
   ('tanque_01', 37.8)
   ('tanque_01', 37.4)
   ('tanque_02', 38.7)
   ('tanque_02', 38.4)
   ('tanque_03', 32.2)
   ('tanque_03', 15.2)
   ('tanque_04', 38.1)
   ('tanque_04', 37.3)


In [ ]:
spark.stop()